In [4]:
# EfficientNet-B0 - Data Preparation

import os
import random
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Paths
PROJECT_ROOT = "/Users/mirshad/Desktop/SLIIT/brain-tumor/brain-tumor-vit"
DATASET_ROOT = os.path.join(PROJECT_ROOT, "data")

TRAIN_DIR = "/Users/mirshad/Desktop/SLIIT/brain-tumor/brain-tumor-vit/archive/Training"
TEST_DIR = "/Users/mirshad/Desktop/SLIIT/brain-tumor/brain-tumor-vit/archive/Testing"

# Configuration
IMG_SIZE = 224
BATCH_SIZE = 8
NUM_CLASSES = 4
NUM_WORKERS = 0

CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]

print("PyTorch version:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())
print("Training directory:", TRAIN_DIR)
print("Testing directory:", TEST_DIR)

PyTorch version: 2.8.0
MPS available: True
Training directory: /Users/mirshad/Desktop/SLIIT/brain-tumor/brain-tumor-vit/archive/Training
Testing directory: /Users/mirshad/Desktop/SLIIT/brain-tumor/brain-tumor-vit/archive/Testing


In [5]:
# Image preprocessing and augmentation

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(7),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Load datasets
full_train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=eval_transform
)

print("Classes:", full_train_dataset.classes)
print("Training images:", len(full_train_dataset))
print("Testing images:", len(test_dataset))


Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Training images: 5600
Testing images: 1600


In [3]:
import os

base = "/Users/mirshad/Desktop/SLIIT/brain-tumor"

for root, dirs, files in os.walk(base):
    if "Training" in dirs or "Testing" in dirs:
        print("Found:", root)
        if "Training" in dirs:
            print("  Training:", os.path.join(root, "Training"))
        if "Testing" in dirs:
            print("  Testing:", os.path.join(root, "Testing"))

Found: /Users/mirshad/Desktop/SLIIT/brain-tumor/brain-tumor-vit/archive
  Training: /Users/mirshad/Desktop/SLIIT/brain-tumor/brain-tumor-vit/archive/Training
  Testing: /Users/mirshad/Desktop/SLIIT/brain-tumor/brain-tumor-vit/archive/Testing


In [6]:
# Create stratified train/validation split

targets = np.array(full_train_dataset.targets)

train_indices, val_indices = train_test_split(
    np.arange(len(targets)),
    test_size=0.20,
    random_state=SEED,
    stratify=targets
)

# Separate transforms for training and validation
train_dataset_full = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

val_dataset_full = datasets.ImageFolder(
    TRAIN_DIR,
    transform=eval_transform
)

train_dataset = Subset(train_dataset_full, train_indices)
val_dataset = Subset(val_dataset_full, val_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))
print("Batch size:", BATCH_SIZE)

Train: 4480
Validation: 1120
Test: 1600
Batch size: 8


In [7]:
# EfficientNet-B0 model setup

import timm

# Device
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("Using device:", DEVICE)

# Load pretrained EfficientNet-B0
model = timm.create_model(
    "efficientnet_b0",
    pretrained=True,
    num_classes=NUM_CLASSES
)

model = model.to(DEVICE)

# Freeze the backbone for Stage 1
for param in model.parameters():
    param.requires_grad = False

# Train only the final classifier
for param in model.classifier.parameters():
    param.requires_grad = True

# Parameter counts
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Model: EfficientNet-B0")
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Classifier:", model.classifier)

Using device: mps


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Model: EfficientNet-B0
Total parameters: 4012672
Trainable parameters: 5124
Classifier: Linear(in_features=1280, out_features=4, bias=True)
